# Notebook 01 — EDA & Benchmark Models (Parts 1–2)

**Run this notebook first.**

Loads German electricity data from Google Drive, performs exploratory analysis, stationarity tests, and benchmark forecasts (Mean, Naive, Seasonal Naive, Drift).

All outputs are saved to Drive for later notebooks.

In [1]:
# --- Setup ---
!pip install -q pandas numpy matplotlib seaborn statsmodels scikit-learn requests holidays tqdm

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import importlib.util
import shutil
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path("/content/drive/MyDrive/Assignment")
UTILS_SRC = PROJECT_ROOT / "colab" / "utils.py"

print("utils.py exists:", UTILS_SRC.exists())
print("utils.py size:", UTILS_SRC.stat().st_size if UTILS_SRC.exists() else "MISSING")

# Copy off Drive into local Colab disk (most reliable)
shutil.copy2(UTILS_SRC, "/content/utils.py")

def _load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

_utils = _load_module("utils", "/content/utils.py")
globals().update({k: getattr(_utils, k) for k in dir(_utils) if not k.startswith("_")})

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

paths = get_paths(PROJECT_ROOT)
csv_path = paths["root"] / DATA_CSV
print("Loaded utils OK")
print("CSV path:", csv_path)
print("CSV exists:", csv_path.exists())

Mounted at /content/drive
utils.py exists: True
utils.py size: 20348
Loaded utils OK
CSV path: /content/drive/MyDrive/Assignment/time_series_60min_singleindex.csv
CSV exists: True


## Part 1 — Load & prepare data

In [2]:
# Load hourly German load (MW)
hourly = load_hourly_load(csv_path)
daily, weekly = aggregate_load(hourly)

print(f"Hourly observations: {len(hourly):,}")
print(f"Daily observations:  {len(daily):,}")
print(f"Weekly observations: {len(weekly):,}")
print(f"Date range: {weekly.index.min().date()} to {weekly.index.max().date()}")

# Save processed series (named index for reliable CSV reload)
hourly.index.name = "date"
daily.index.name = "date"
weekly.index.name = "date"
hourly.to_csv(paths["processed"] / "hourly_load_mw.csv")
daily.to_csv(paths["processed"] / "daily_load_gw.csv")
weekly.to_csv(paths["processed"] / "weekly_load_gw.csv")

Hourly observations: 50,400
Daily observations:  2,100
Weekly observations: 301
Date range: 2015-01-04 to 2020-10-04


In [3]:
# Initial time series plots
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

axes[0].plot(hourly.index, hourly / 1000, linewidth=0.3, alpha=0.7)
axes[0].set_title("Hourly German electricity load")
axes[0].set_ylabel("GW")

axes[1].plot(daily.index, daily, linewidth=0.8)
axes[1].set_title("Daily average load")
axes[1].set_ylabel("GW")

axes[2].plot(weekly.index, weekly, linewidth=1.5, color="darkblue")
axes[2].set_title("Weekly average load (modelling series)")
axes[2].set_ylabel("GW")
axes[2].set_xlabel("Date")

plt.tight_layout()
plot_and_save(fig, paths["figures"] / "01_timeseries_overview.png")
plt.show()

In [4]:
# STL decomposition — weekly series (annual seasonality period = 52)
stl = STL(weekly, period=SEASONALITY, robust=True)
result = stl.fit()

fig = result.plot()
fig.set_size_inches(12, 9)
fig.suptitle("STL decomposition of weekly load (period=52)", y=1.02)
plot_and_save(fig, paths["figures"] / "02_stl_decomposition.png")
plt.show()

# ACF / PACF
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

plot_acf(weekly, lags=60, ax=axes[0, 0])
axes[0, 0].set_title("ACF — weekly load")

plot_pacf(weekly, lags=60, ax=axes[0, 1], method="ywm")
axes[0, 1].set_title("PACF — weekly load")

weekly_diff = weekly.diff().dropna()
plot_acf(weekly_diff, lags=60, ax=axes[1, 0])
axes[1, 0].set_title("ACF — first difference")

plot_pacf(weekly_diff, lags=60, ax=axes[1, 1], method="ywm")
axes[1, 1].set_title("PACF — first difference")

plt.tight_layout()
plot_and_save(fig, paths["figures"] / "03_acf_pacf.png")
plt.show()

In [5]:
# Stationarity tests (ADF + KPSS)
tests = pd.concat([
    run_stationarity_tests(weekly, "weekly_level"),
    run_stationarity_tests(weekly_diff, "weekly_diff1"),
    run_stationarity_tests(weekly.diff(SEASONALITY).dropna(), "weekly_seasonal_diff"),
], ignore_index=True)

print(tests.round(4).to_string(index=False))
tests.to_csv(paths["metrics"] / "01_stationarity_tests.csv", index=False)

print("\nInterpretation:")
print("- Strong annual seasonality visible in ACF at lag 52")
print("- Raw weekly series is non-stationary (trend + seasonality)")
print("- Differencing (d=1, D=1) is appropriate for SARIMA")

              series test  statistic  p_value  crit_5pct  stationary_at_5pct
        weekly_level  ADF    -4.0475   0.0012    -2.8715                True
        weekly_level KPSS     0.1604   0.1000     0.4630                True
        weekly_diff1  ADF    -7.0693   0.0000    -2.8715                True
        weekly_diff1 KPSS     0.0558   0.1000     0.4630                True
weekly_seasonal_diff  ADF    -4.2951   0.0005    -2.8734                True
weekly_seasonal_diff KPSS     1.3419   0.0100     0.4630               False

Interpretation:
- Strong annual seasonality visible in ACF at lag 52
- Raw weekly series is non-stationary (trend + seasonality)
- Differencing (d=1, D=1) is appropriate for SARIMA


## Part 2 — Benchmark models (2-year horizon)

In [6]:
# Train-test split (last 104 weeks = 2 years)
train, test = train_test_split_series(weekly, TEST_WEEKS)
print(f"Train: {train.index.min().date()} → {train.index.max().date()} ({len(train)} weeks)")
print(f"Test:  {test.index.min().date()} → {test.index.max().date()} ({len(test)} weeks)")

# Benchmark forecasts (no data leakage)
benchmarks = run_benchmarks(train, test)

metrics = []
for name, pred in benchmarks.items():
    metrics.append(evaluate_forecast(name, test, pred, train))

metrics_df = save_metrics(metrics, paths["metrics"] / "02_benchmark_metrics.csv")
print("\nBenchmark model comparison:")
print(metrics_df.round(3).to_string(index=False))

Train: 2015-01-04 → 2018-10-07 (197 weeks)
Test:  2018-10-14 → 2020-10-04 (104 weeks)

Benchmark model comparison:
         model   MAE  RMSE  MASE   Bias
seasonal_naive 2.319 3.007 1.732  1.732
         naive 3.783 4.459 2.827 -0.882
          mean 3.789 4.397 2.831  0.481
         drift 4.340 5.118 3.243  1.007


In [7]:
# Benchmark forecast plot
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(train.index, train, label="Training", color="steelblue", linewidth=1.2)
ax.plot(test.index, test, label="Test (actual)", color="black", linewidth=2)

colors = {"mean": "gray", "naive": "orange", "seasonal_naive": "green", "drift": "purple"}
for name, pred in benchmarks.items():
    ax.plot(test.index, pred, linestyle="--", label=name.replace("_", " ").title(),
            color=colors.get(name, None), linewidth=1.5)

ax.set_title("Benchmark forecasts — weekly German electricity demand")
ax.set_ylabel("Average load (GW)")
ax.set_xlabel("Date")
ax.legend()
plt.tight_layout()
plot_and_save(fig, paths["figures"] / "04_benchmark_forecasts.png")
plt.show()

## Build feature table (temperature + holidays) for later notebooks

In [8]:
# Fetch temperature & holiday features (saved for notebooks 03+)
feature_df = build_weekly_feature_table(weekly, paths)
print("Feature table shape:", feature_df.shape)
print(feature_df.head())
print("\nSaved to:", paths["processed"] / "weekly_features.csv")

# Save train/test indices for downstream notebooks
save_json({
    "train_end": str(train.index[-1]),
    "test_start": str(test.index[0]),
    "test_weeks": TEST_WEEKS,
    "seasonal_order": [1, 1, 1, SEASONALITY],
}, paths["checkpoints"] / "01_metadata.json")

free_memory()
print("\n✓ Notebook 01 complete. Proceed to 02_SARIMA_GridSearch.ipynb")

Feature table shape: (301, 8)
              load_gw  temp_mean  temp_min  temp_max  heating_degree  \
date                                                                   
2015-01-04  47.233740   3.000000       3.0       3.0            12.5   
2015-01-11  56.191101   3.885714       1.2       8.5            81.3   
2015-01-18  57.672679   4.900000      -0.8       9.2            74.2   
2015-01-25  58.613304   0.028571      -0.7       0.9           108.3   
2015-02-01  58.734030   1.414286      -0.1       2.8            98.6   

            cooling_degree  holiday_days  has_holiday  
date                                                   
2015-01-04             0.0             1            1  
2015-01-11             0.0             0            0  
2015-01-18             0.0             0            0  
2015-01-25             0.0             0            0  
2015-02-01             0.0             0            0  

Saved to: /content/drive/MyDrive/Assignment/data/processed/weekly_featur